In [30]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
from plotly.subplots import make_subplots
pd.set_option('display.max_columns', None)


In [31]:
#Read in AWI dataset
gdf_final = gpd.read_file("O:/NaturalEngland/Ancient Woodland Inventory Update/_Deliverable/Final/AWI_final.shp")
gdf_methodology = gpd.read_file("O:/NaturalEngland/Ancient Woodland Inventory Update/_Deliverable/Final/AWI_methodology_layer.shp")

In [102]:
# All file names and locations for designation sites
file_list = { 'SSSI':{'file_location':"G:/NE_OpenData/SSSI/NE_Sites_of_Special_Scientific_Interest_Wilts.shp",
         'name':"SSSI",
           'fullName':'Sites of Special Scientific Interest',
         'type':'polygon'},
'NNR':{'file_location':"G:/NE_OpenData/NNR/NE_National_Nature_Reserves_Wilts.shp",
         'name':"NNR",
           'fullName':'National Nature Reserves',
         'type':'polygon'},
        'LNR': {'file_location':"G:/NE_OpenData/LNR/NE_Local_Nature_Reserves_Wilts.shp",
         'name':"LNR",
           'fullName':'Local Nature Reserves',
         'type':'polygon'},
         'RAMSAR':{'file_location':"G:/NE_OpenData/RAMSAR/NE_RAMSAR_New_Forest.shp",
         'name':"RAMSAR",
           'fullName':'RAMSAR',
         'type':'polygon'},
         'SPA':{'file_location':"G:/NE_OpenData/SPA/NE_Special_Protection_Areas_iWilts.shp",
         'name':"SPA",
           'fullName':'Special Protection Areas',
         'type':'polygon'},
         'SAC':{'file_location':"G:/NE_OpenData/SAC/NE_Special_Areas_of_Conservation_Wilts.shp",
         'name':"SAC",
           'fullName':'Special Areas of Conservation',
         'type':'polygon'},
         'LCA':{'file_location':"G:/Wiltshire_Council/Landscape_Character_Areas/LCA_Combined_2005.shp",
         'name':"LCA",
           'fullName':'Landscape Character Areas',
         'type':'polygon'},
         'CP': {'file_location':"G:/NE_OpenData/Country_Parks/NE_Country_Parks_Wilts.shp",
         'name':"CP",
           'fullName':'Country Parks',
         'type':'polygon'},
         'LB':{'file_location':"G:/Historic_England/ListedBuildings/HEListed_Building_pts_Wilts.shp",
         'name':"LB",
           'fullName':'Listed Buildings',
         'type':'point'},
         'SAM':{'file_location':"G:/Historic_England/ScheduledMonuments/HE_Scheduled_Monuments_Wilts.shp",
         'name':"SM",
           'fullName':'Scheduled Monuments',
         'type':'point'},
         'NL': {'file_location':"G:/NE_OpenData/National_Landscapes/NE_National_Landscapes_iWilts.shp",
         'name':"NL",
           'fullName':'National Landscape',
         'type':'polygon'},
         'ATI':{'file_location':"G:/Woodland_Trust/Woodland_Trust_AncientTreeInventoryATI_Wilts.shp",
         'name':"ATI",
           'fullName':'Ancient Tree Inventory',
         'type':'polygon'},
         'WT':{'file_location':"G:/Woodland_Trust/Woodland_Trust_Sites_Wilts.shp",
         'name':"WT",
           'fullName':'Woodland Trust Sites',
         'type':'polygon'},
         'WWT':{'file_location':"G:/WWT/WWT_Reserves_Public_201224/WiltshireWildlifeTrust_Reserves_Public_201224.shp",
         'name':"WWT",
           'fullName':'Wiltshire Wildlife Trust Reserves',
         'type':'polygon'},
         'AWI': {'file_location':"G:/NE_OpenData/AWI/Old_AWI_Wilts.shp",
         'name':"ASNW_OLD",
           'fullName':'Previous Ancient Woodland Inventory',
         'type':'polygon'},
          'NCA': {'file_location':"G:/NE_OpenData/National_Character_Areas/NE_National_Character_Areas_Wilts.shp",
         'name':"NCA",
        'fullName':'National Character Areas',
         'type':'polygon'},
         'BCZ':{'file_location':"G:/Wiltshire_Council/Bat_Consultation_Zones/BatConsultationZones.shp",
         'name':"BCZ",
         'fullName':'Bat Consultation Zones',
         'type':'polygon'},
         'BAP':{'file_location':"G:/WSBRC_Data/BAP_LandscapeBiodiversityAreas/WSBRC_LandscapeBiodiversityAreas.shp",
         'name':"BAP",
         'fullName':'BAP_LandscapeBiodiversityAreas',
         'type':'polygon'},
         'NT':{'file_location':"G:/National_Trust/NT_Land_Ownership_Wilts.shp",
         'name':"NT",
         'fullName':'National Trust',
         'type':'polygon'},
          'CWS':{'file_location':"G:/WSBRC_Data/Sites/CWS.shp",
         'name':"CWS",
         'fullName':'County Wildlife Sites',
         'type':'polygon'},
         
}

rpa_file = {
    'file_locations':"G:/RPA/RPA_Parcels_20241218.shp",
    'name':'RPA',
    'type':'polygon'
}


In [ ]:
# Other options for analysis
## intersected_layer = gpd.overlay(gdf_methodology, gdf_other, how='intersection')
## join_layer = gdf_methodology.sjoin(gdf_other, how = 'left',predicate='intersects' )

C:\Users\WSBRCvol\AppData\Local\Temp\ipykernel_10300\416847333.py:1: UserWarning: `keep_geom_type=True` in overlay resulted in 13 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersected_layer = gpd.overlay(gdf_methodology, gdf_other, how='intersection')


In [33]:
# Create new column to identify Long established woodland, which is not ancient woodland
def new_status( row):
    if row['ACTION'] == 'EXCLUDE':
        return 'LEW'
    else:
        return row['Status']


In [34]:
# Add column which shows LEW
gdf_methodology['Status_updated'] = gdf_methodology.apply(new_status, axis =1)

In [35]:
# Check parcels which intersect with each designation dataset
def check_intersections(file):
    gdf_other = gpd.read_file(file['file_location'])
    gdf_methodology[f'{file['name']}_i'] = gdf_methodology.geometry.apply(lambda x: gdf_other.geometry.intersects(x).any())
    return gdf_methodology

In [36]:
# Loop through each designation file
for file in file_list:
    check_intersections(file_list[file])

In [112]:
gdf_methodology['total_designations'] = gdf_methodology[['SSSI_i', 'NNR_i', 'LNR_i', 'RAMSAR_i', 'SPA_i', 'SAC_i', 
       'CP_i', 'LB_i', 'SM_i', 'NL_i', 'ATI_i', 'WT_i', 'WWT_i', 'ASNW_OLD_i',
       'BCZ_i',  'NT_i', 'CWS_i']].sum(axis=1)

In [123]:



for file in file_list:
    # print(file)
    gpd_desig = gpd.read_file(file_list[file]['file_location'])
    intersection_desig_AWI = gpd.overlay(gdf_methodology, gpd_desig, how="intersection", keep_geom_type=False)
    print(file)
    print(intersection_desig_AWI.area.sum()/10000)
    print((100*intersection_desig_AWI.area.sum())/(gdf_methodology.area.sum()))

SSSI
3674.358054187738
14.800438573889995
NNR
207.82588100563802
0.8371296810289998
LNR
108.72364777499511
0.4379425321898279
RAMSAR
1.5437730307815816
0.006218370005631291
SPA
596.5558207393283
2.4029470319818307
SAC
1006.9945564158287
4.056208147566822
LCA
23499.289468271112
94.65593313880024
CP
154.63812778945368
0.6228876113259214
LB
0.0
0.0
SAM
253.4991703646753
1.0211032360437324
NL
14670.672564300705
59.093965510006775
ATI
0.0
0.0
WT
169.27240107383696
0.6818349593047549
WWT
225.17529223476626
0.9070136965231385
AWI
12262.498762741303
49.393759950396394
NCA
24698.755842478477
99.4874242812243
BCZ
5867.590528327745
23.63485319354682
BAP
24695.12373693122
99.47279403717806
NT
309.8848941117203
1.2482268392569076
CWS
11284.72771148342
45.455265070460364


In [129]:
summary_list_desig = []
for file in file_list:
    gpd_desig = gpd.read_file(file_list[file]['file_location'])
    intersection_desig_AWI = gpd.overlay(gdf_methodology, gpd_desig, how="intersection", keep_geom_type=False)
    summary_list_desig.append([
        file_list[file]['fullName'],
        intersection_desig_AWI.area.sum()/10000,
       (intersection_desig_AWI.area.sum())/(gdf_methodology.area.sum())
        
        ])

In [126]:
df_desig=pd.DataFrame(summary_list_desig,columns=['Designation',  'AreaHa',  'Percentage_area'])

In [127]:
df_desig

,Designation,AreaHa,Percentage_area
0,Sites of Special Scientific Interest,3674.358054,14.800439
1,National Nature Reserves,207.825881,0.837130
2,Local Nature Reserves,108.723648,0.437943
3,RAMSAR,1.543773,0.006218
4,Special Protection Areas,596.555821,2.402947
5,Special Areas of Conservation,1006.994556,4.056208
6,Landscape Character Areas,23499.289468,94.655933
7,Country Parks,154.638128,0.622888
8,Listed Buildings,0.000000,0.000000
9,Scheduled Monuments,253.499170,1.021103


In [128]:
df_desig.to_csv("C:/Users/Lara/AWI/Data/Designation_area_AWI_area_correct.csv")

In [118]:
desig_list = ['SSSI_i', 'NNR_i', 'LNR_i', 'RAMSAR_i', 'SPA_i', 'SAC_i', 
       'CP_i', 'LB_i', 'SM_i', 'NL_i', 'ATI_i', 'WT_i', 'WWT_i', 'ASNW_OLD_i',
       'BCZ_i',  'NT_i', 'CWS_i']
for desig in desig_list:
    print(desig)
    print(100*gdf_methodology[gdf_methodology[desig] ==1].AreaHa.sum()/gdf_methodology.AreaHa.sum())
    print(gdf_methodology[gdf_methodology[desig]==1].shape[0])

SSSI_i
22.762471917624993
978
NNR_i
1.3530246028734703
29
LNR_i
0.44996338572631656
31
RAMSAR_i
0.4599141944254181
10
SPA_i
2.9533857296222465
235
SAC_i
6.2397766759893925
412
CP_i
0.7370703630101607
35
LB_i
3.112749053301878
64
SM_i
15.381922506866562
365
NL_i
59.229451622251176
3882
ATI_i
14.755537143915722
233
WT_i
0.928184451989511
41
WWT_i
1.4962978205425976
78
ASNW_OLD_i
57.48323280720803
2096
BCZ_i
25.406331194455763
1584
NT_i
2.1141362958655874
130
CWS_i
60.32946348378426
2629


In [ ]:
# Write out dataset
#gdf_methodology.to_file("O:/NaturalEngland/Ancient Woodland Inventory Update/_Deliverable/Prep/summary_stats/AWI_methodology_v2.shp")

C:\Users\WSBRCvol\AppData\Local\Temp\ipykernel_10300\706631678.py:2: UserWarning:

Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.

C:\Users\WSBRCvol\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\raw.py:723: RuntimeWarning:

Normalized/laundered field name: 'Status_updated' to 'Status_upd'



## Graphs for designation areas

In [37]:
# Show number of parcels in designation (true) and outside designation (false)
fig = make_subplots(rows=1, cols=20,subplot_titles=list(file_list.keys()),shared_yaxes=True) 
x=1

for file in file_list:
    fig.add_trace(go.Histogram(x = gdf_methodology[f"{file_list[file]['name']}_i"]),row = 1, col = x)
    # fig.update_yaxes(title_text=name, row=1, col=x)
    fig.update_layout(template = 'plotly_white',showlegend=False, bargap=0.2)
    x+=1

fig.show()

## Summary stats

In [299]:
## Summary statistics
gdf_totals =  gdf_methodology.groupby('Status_updated')['AreaHa'].sum().reset_index()
gdf_totals['area_percent'] = gdf_totals.AreaHa/gdf_totals.AreaHa.sum()

In [300]:
#Total area excluding LEW
gdf_AW = gdf_methodology[gdf_methodology['Status_updated']!= "LEW"]
gdf_AW.AreaHa.sum()

np.float64(22781.57078414469)

In [46]:
gdf_final.groupby('Status')['AreaHa'].sum()

Status
ASNW     6823.668384
AWP      2980.823987
IAWP     2885.125673
PAWS    10081.672284
Name: AreaHa, dtype: float64

In [90]:
# Parcel size by status
print(gdf_methodology.groupby('Status_updated')['AreaHa'].sum())
# Parcel size by status percent
print(gdf_methodology.groupby('Status_updated')['AreaHa'].sum()/gdf_methodology.AreaHa.sum())

Status_updated
ASNW     6823.668384
AWP      2980.823987
IAWP     2885.125673
LEW      2044.437147
PAWS    10081.672284
Name: AreaHa, dtype: float64
Status_updated
ASNW    0.274860
AWP     0.120069
IAWP    0.116214
LEW     0.082351
PAWS    0.406093
Name: AreaHa, dtype: float64


## Designation areas summary status

In [108]:
summary_list = []
for file in file_list:
    summary_list.append([
        file_list[file]['fullName'],
       # gdf_methodology.groupby(f"{file_list[file]['name']}_i")['AreaHa'].count()[1],
        gdf_methodology.groupby(f"{file_list[file]['name']}_i")['AreaHa'].sum()[1].round(),
       # (100*gdf_methodology.groupby(f"{file_list[file]['name']}_i")['AreaHa'].count()[1])/6990,
        ((100*gdf_methodology.groupby(f"{file_list[file]['name']}_i")['AreaHa'].sum()[1])/24826.00793107435).round(),
        
        ])

C:\Users\Lara\AppData\Local\Temp\ipykernel_18080\656053723.py:6: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Lara\AppData\Local\Temp\ipykernel_18080\656053723.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [109]:
df=pd.DataFrame(summary_list,columns=['Designation',  'AreaHa',  'Percentage_area'])

In [116]:
gdf_methodology.groupby('total_designations')['AreaHa'].sum()

total_designations
0    2130.561454
1    3721.268427
2    4616.939767
3    6681.477845
4    4820.730784
5    1645.817817
6     656.662104
7     552.549732
Name: AreaHa, dtype: float64

In [111]:
df.to_csv("C:/Users/Lara/AWI/Data/Designation_area_AWI.csv")

## National Character Areas analysis

In [39]:
gdf_nca = gpd.read_file(file_list['NCA']['file_location'])

In [ ]:
gdf_nca_intersect = gpd.overlay(gdf_methodology, gdf_nca, how='intersection')
gdf_nca_intersect['area'] = (gdf_nca_intersect.area*0.0001)
gdf_nca_intersect['area_percent'] =( gdf_nca_intersect['area']/24826.00793107435)
gdf_nca_intersect['area']  = gdf_nca_intersect['area'].round(1)
gdf_nca_joined_grouped = gdf_nca_intersect.groupby(['NANAME','Status_updated' ])[['area', 'area_percent']].sum().reset_index()

In [72]:
gdf_pivot = gdf_nca_joined_grouped.pivot(index ='NANAME' ,columns=[ 'Status_updated'], values =[ 'area', 'area_percent']).reset_index()
#gdf_pivot.to_csv("O:/NaturalEngland/Ancient Woodland Inventory Update/_Deliverable/Prep/summary_stats/AWI_nca_summary.csv")

In [86]:
gdf_pivot

NANAME    area                        \
Status_updated                                     ASNW    AWP    IAWP    LEW   
0               Berkshire and Marlborough Downs  1007.2  471.3   235.3  350.3   
1                                     Cotswolds   637.3  308.7   143.9  210.3   
2                               Hampshire Downs   253.4  163.9    99.2   41.4   
3                                  London Basin    13.0    NaN     NaN    NaN   
4                                 Midvale Ridge    62.8   24.0     5.8   19.3   
5                                    New Forest   500.9   26.4   434.1   16.1   
6                            South Wessex Downs  2006.1  815.8  1145.6  891.7   
7                         Thames and Avon Vales  1573.5  549.3   252.3  435.0   
8                                  Wessex Vales   738.1  607.7   560.1   59.7   

                       area_percent                                          
Status_updated    PAWS         ASNW       AWP      IAWP       LEW      PAWS  
0               2266.1     0.040585  0.018969  0.009464  0.014099  0.091265  
1                404.6     0.025646  0.012426  0.005779  0.008474  0.016287  
2                324.2     0.010211  0.006608  0.003995  0.001670  0.013058  
3                  2.1     0.000520       NaN       NaN       NaN  0.000086  
4                 29.0     0.002524  0.000968  0.000232  0.000774  0.001169  
5                315.4     0.020169  0.001066  0.017481  0.000648  0.012702  
6               3597.6     0.080798  0.032885  0.046118  0.035937  0.144903  
7               1278.8     0.063380  0.022106  0.010180  0.017507  0.051505  
8               1815.4     0.029702  0.024476  0.022552  0.002412  0.073123

## BAP landscape biodiversity areas

In [38]:
gdf_BAP = gpd.read_file(file_list['BAP']['file_location'])

In [275]:
gdf_BAP_intersect = gpd.overlay(gdf_methodology, gdf_BAP, how='intersection')
gdf_BAP_intersect['area'] = (gdf_BAP_intersect.area*0.0001)
gdf_BAP_intersect['area_percent'] =( gdf_BAP_intersect['area']/24826.00793107435)
gdf_BAP_intersect['area']  = gdf_BAP_intersect['area'].round(1)
gdf_BAP_intersect = gdf_BAP_intersect.groupby(['LBA_Name','Status_updated' ])[['area', 'area_percent']].sum().reset_index()

In [276]:
gdf_pivot = gdf_BAP_intersect.pivot(index ='LBA_Name' ,columns=[ 'Status_updated'], values =[ 'area', 'area_percent']).reset_index()

## Existing AWI analysis 

In [ ]:
# Read in old AWI
gdf_awi_old = gpd.read_file("G:/NE_OpenData/AWI/Old_AWI_Wilts.shp")
gdf_awi_old.groupby('status')['area'].sum()

status
ASNW    6327.666172
PAWS    7544.051754
Name: area, dtype: float64

In [ ]:
# read in old wood pasture/ parkland data. 
## Ignored this in the end as data was ropey, large areas covered which were not parkland
gdf_wp_old = gpd.read_file("G:/NE_OpenData/Wood_Pasture_and_Parkland/NE_Wood_Pasture_Parkland_Wilts.shp")
gdf_wp_old['areaHa'] = gdf_wp_old.area*0.0001
gdf_wp_wc_removed = gdf_wp_old[gdf_wp_old.SUBTYPE != 'Wooded Common']


In [56]:
gdf_awi_old['areaHa'] = gdf_awi_old.area * 0.0001

In [ ]:
# Summary stats for old AWI
gdf_awi_old.groupby('status')['area'].sum()

status
ASNW    6327.666172
PAWS    7544.051754
Name: area, dtype: float64

In [ ]:
# Summary stats for new AWI
gdf_final.groupby('Status')['AreaHa'].sum()

Status
ASNW     6823.668384
AWP      2980.823987
IAWP     2885.125673
PAWS    10081.672284
Name: AreaHa, dtype: float64

In [61]:
gdf_new_old_all = gdf_methodology.sjoin(gdf_awi_old, predicate = 'intersects', how = 'left')
gdf_new_old_only = gdf_methodology.sjoin(gdf_awi_old, predicate = 'intersects', how = 'inner')
gdf_old_new_only =  gdf_awi_old.sjoin(gdf_methodology, predicate = 'intersects', how = 'left')

In [62]:
print(gdf_methodology.shape[0])
print(gdf_awi_old.shape[0])
print(gdf_new_old_all.shape[0])
print(gdf_new_old_only.shape[0])
print(gdf_old_new_only.shape[0])

6990
1193
8009
3115
3136


In [65]:
# Add column to show old and new status
gdf_new_old_all['status_combined'] = "Old :"+ gdf_new_old_all.status.fillna('None') + '; New :'+gdf_new_old_all.Status_updated.fillna('None') 

In [66]:
# Plot column showing old/ new status
fig = px.histogram(gdf_new_old_all, 'status_combined')
fig.show()

In [67]:
# Find area of intersection for old and new AWI 
intersection_old_new = gpd.overlay(gdf_methodology, gdf_awi_old, how="intersection", keep_geom_type=False)
intersection_old_new['status_combined'] = "Old :"+ intersection_old_new.status + '; New :'+intersection_old_new.Status_updated 
intersection_old_new['area_intersect'] = intersection_old_new.area/10000
intersection_old_new_grouped =  intersection_old_new.groupby('status_combined')['area_intersect'].sum().reset_index()

In [68]:
fig = px.bar(intersection_old_new_grouped, 'status_combined', 'area_intersect', color = 'status_combined')
fig.show()

In [ ]:
# New PAWS sites (PAWS -( Old: ASNW; New: PAWS  + Old: Paws ; New PAWS ) )
10081 - (1648+5933)

2500

In [106]:
## Check AWI old parcel size compared to new parcel size
print(gdf_final.area.mean()/10000)
print(gdf_awi_old.area.mean()/10000)

4.083450579699739
11.627592561501624


## RPA analysis

In [ ]:
rpa_gdf = gpd.read_file(rpa_file['file_locations'])

#gdf_rpa_test = gdf_methodology.left_join()